<a href="https://colab.research.google.com/github/Ridd1kulusC0d3r/OSINT-Colab/blob/main/notebooks/07_coleta_fontes_abertas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir no Colab"/></a>

# 07 · Coleta em fontes abertas para alimentar a IA

**OSINTIA** · modelos de IA para OSINT, reconstruídos e auditáveis · [repositório](https://github.com/Ridd1kulusC0d3r/OSINT-Colab)

O elo que faltava: **coletar** antes de analisar. Partindo de uma marca ou username, este notebook gera variantes de imitação, procura domínios em certificados públicos (**crt.sh**), monta a linha do tempo de **IDs de analytics** no Wayback Machine (técnica da Bellingcat) e verifica contas com **Maigret** e **Holehe**. A saída já sai no formato dos notebooks 01 (triagem) e 04 (vínculos).

| | |
|---|---|
| **Hardware** | CPU (não precisa de GPU) |
| **Tempo estimado** | 5 a 20 minutos (depende das fontes) |
| **Como rodar** | Menu *Ambiente de execução* → *Executar tudo* |

> ⚖️ Uso ético: só fontes abertas, finalidade legítima e respeito à LGPD. Os dados de demonstração são **fictícios**. Leia `docs/METODOLOGIA.md`.

> ⚠️ **Antes de rodar com um alvo real:** confirme a finalidade legítima (proteção de marca, resposta a incidente, investigação autorizada). Prefira alvos **institucionais** (marcas, domínios). Para pessoas físicas, registre a base legal (LGPD) no manifesto. Holehe consulta serviços com o e-mail informado: use só com e-mails sob sua responsabilidade ou com autorização.

In [ ]:
# Baixa o OSINTIA (núcleo em Python puro: calibração, auditoria, roteamento, custódia)
!git clone -q https://github.com/Ridd1kulusC0d3r/OSINT-Colab.git /content/OSINT-Colab 2>/dev/null || git -C /content/OSINT-Colab pull -q
import sys
sys.path.insert(0, "/content/OSINT-Colab")
from osintia import auditoria, calibracao, coleta, coordenacao, custodia, dados, esquemas, extracao, geo, roteamento
print("OSINTIA pronto")

## 0. Defina o alvo
O padrão usa o caso fictício. Troque pelos dados do seu caso.

In [ ]:
MARCA         = "bancoaurora"                     # username oficial / nome da marca
TERMO_CRTSH   = "%aurora%"                        # % é curinga no crt.sh
SITE_OFICIAL  = "aurora.example"                  # domínio oficial (para comparar analytics)
SUSPEITOS     = []                                # domínios suspeitos já conhecidos (opcional)
EMAIL         = ""                                # e-mail sob sua responsabilidade (opcional, para o Holehe)
MAX_VARIANTES_MAIGRET = 15                        # Maigret é lento: comece com poucas variantes
BASE_LEGAL    = "proteção de marca - contrato com o cliente"

import os, json
os.makedirs("/content/saidas", exist_ok=True)
MANIFESTO = "/content/saidas/manifesto.jsonl"
custodia.registrar(MANIFESTO, "inicio_coleta", detalhes={"marca": MARCA, "base_legal": BASE_LEGAL})

## 1. Variantes de imitação (lógica do SÓSIA)
Homóglifos (`o→0`), letra dobrada ou omitida, letras trocadas, separadores e sufixos típicos de golpe (`_oficial`, `_sac`, `_suporte`).

In [ ]:
vars_ = coleta.variantes(MARCA)
print(len(vars_), "variantes. Primeiras 25:")
print(vars_[:25])

## 2. Domínios em certificados públicos (crt.sh)
Todo certificado HTTPS emitido vai para logs públicos de *Certificate Transparency*. Um domínio imitador costuma aparecer aqui **antes** de o golpe ir ao ar.

In [ ]:
try:
    certs = coleta.crtsh(TERMO_CRTSH)
except Exception as e:
    certs = []
    print("crt.sh indisponível agora (é comum ficar lento). Tente de novo em alguns minutos:", type(e).__name__)

oficiais = {SITE_OFICIAL, "www." + SITE_OFICIAL}
suspeitos_ct = sorted({c["dominio"] for c in certs if c["dominio"] not in oficiais and not c["dominio"].endswith("." + SITE_OFICIAL)})
print(len(certs), "domínios em certificados;", len(suspeitos_ct), "fora do domínio oficial")
for c in sorted(certs, key=lambda c: c["inicio"], reverse=True)[:15]:
    print(c["inicio"][:10], c["dominio"], "|", c["emissor"][:40])

with open("/content/saidas/crtsh.json", "w", encoding="utf-8") as f:
    json.dump(certs, f, ensure_ascii=False, indent=1)
custodia.registrar(MANIFESTO, "coleta_crtsh", "/content/saidas/crtsh.json", {"termo": TERMO_CRTSH})

## 3. IDs de analytics no Wayback Machine
Sites do mesmo dono costumam reaproveitar o mesmo ID (`UA-`, `G-`, `GTM-`, pixel do Facebook, AdSense). Um ID igual entre um site de golpe e outro site é um **vínculo de infraestrutura**: pista forte, mas não prova de autoria.

In [ ]:
alvos = [SITE_OFICIAL] + list(SUSPEITOS) + suspeitos_ct[:5]
historicos = {}
for site in alvos:
    try:
        historicos[site] = coleta.historico_analytics(site, max_capturas=8, de="2018")
        ids = {i for cap in historicos[site] for t, v in cap["ids"].items() if t != "erro" for i in v}
        print(f"{site}: {len(historicos[site])} capturas, IDs: {sorted(ids) or '-'}")
    except Exception as e:
        print(f"{site}: sem capturas ({type(e).__name__})")

compartilhados = coleta.ids_compartilhados(historicos)
print("\nIDs em mais de um site:", compartilhados or "nenhum")
with open("/content/saidas/analytics.json", "w", encoding="utf-8") as f:
    json.dump({"historicos": historicos, "compartilhados": compartilhados}, f, ensure_ascii=False, indent=1)
custodia.registrar(MANIFESTO, "coleta_wayback_analytics", "/content/saidas/analytics.json")

## 4. Contas com Maigret
O Maigret procura um username em milhares de sites e devolve só **perfis públicos que existem**. Aqui testamos o username oficial e as variantes mais próximas. Ferramenta da comunidade (MIT), o mesmo uso do notebook da Bellingcat.

In [ ]:
!pip install -q maigret

In [ ]:
import glob, subprocess
usernames = [MARCA] + vars_[:MAX_VARIANTES_MAIGRET]
pasta = "/content/saidas/maigret"
os.makedirs(pasta, exist_ok=True)
subprocess.run(["maigret", *usernames, "--json", "ndjson", "--folderoutput", pasta, "--timeout", "15", "--no-progressbar"],
               check=False)

perfis = []
for arq in glob.glob(f"{pasta}/*.json"):
    for reg in coleta.ler_ndjson(arq):
        perfis.append({"username": reg.get("username"), "site": reg.get("sitename"),
                       "url": reg.get("url_user"), "arquivo": os.path.basename(arq)})
print(len(perfis), "perfis encontrados")
for p in perfis[:20]:
    print(p["username"], "|", p["site"], "|", p["url"])

## 5. (Opcional) E-mail com Holehe
Só roda se `EMAIL` estiver preenchido.

In [ ]:
if EMAIL:
    !pip install -q holehe
    !cd /content/saidas && holehe --only-used --csv {EMAIL}
    custodia.registrar(MANIFESTO, "coleta_holehe", detalhes={"email": "[mascarado]"})
else:
    print("EMAIL vazio: etapa pulada")

## 6. Entregar para a IA
Cada perfil vira um `state` no mesmo formato da triagem (notebook 01). A similaridade com o username oficial entra como contexto. Campos que a coleta não traz (seguidores, idade da conta) ficam como *desconhecido*: preencha à mão nos casos que for escalar.

In [ ]:
from difflib import SequenceMatcher
estados = []
for i, p in enumerate(perfis):
    sim = SequenceMatcher(None, MARCA, (p["username"] or "").lower()).ratio()
    estados.append({
        "id": f"coleta_{i:04d}",
        "state": "\n".join([
            f"Alvo monitorado: @{MARCA}",
            f"Perfil analisado: @{p['username']} ({p['site']})",
            f"URL: {p['url']}",
            f"Similaridade com o username oficial: {sim:.2f}",
            "Selo de verificado: desconhecido",
            "Seguidores: desconhecido",
        ]),
        "fonte": p["url"],
    })
dados.salvar_jsonl(estados, "/content/saidas/para_triagem.jsonl")
custodia.registrar(MANIFESTO, "entrega_para_triagem", "/content/saidas/para_triagem.jsonl", {"n": len(estados)})
print(len(estados), "perfis prontos para o notebook 01 em /content/saidas/para_triagem.jsonl")
print("Cadeia de custódia:", custodia.verificar(MANIFESTO) or "íntegra")

### Leitura
- **Existir não é imitar.** Maigret acha perfis com o mesmo nome; quem decide se é imitação é a triagem (01) e o analista.
- **crt.sh pode trazer domínios legítimos** de parceiros e subsidiárias. Confira o dono antes de pedir remoção.
- **ID de analytics compartilhado** é vínculo de infraestrutura. Combine com WHOIS, hospedagem e conteúdo antes de concluir.

---
### Próximos passos
- `docs/ENGENHARIA-REVERSA.md` explica como cada peça foi reconstruída.
- `docs/AUDITORIA-DE-ROTULOS.md` mostra por que acurácia alta pode esconder erro herdado.
- `docs/COLABS-RELACIONADOS.md` lista outros notebooks de OSINT (Bellingcat e comunidade) que combinam com este.
- Abra uma *issue* no repositório se algo quebrar no Colab: as bibliotecas de IA mudam rápido.

*OSINTIA · Deivison Lourenço · MIT*